# Exploración: SUNARP — Servicio de Publicidad Registral en Línea (SPRL)

- **Entidad oficial:** Superintendencia Nacional de los Registros Públicos (SUNARP).
- **Descripción y propósito:** Plataforma transaccional autenticada del Estado peruano que permite acceder a los servicios de publicidad registral, búsqueda de índices y visualización de partidas registrales (Personas Jurídicas, Propiedad Inmueble, Mandatos y Poderes, etc.).
- **Enlaces del flujo SSO:**
  - Portada de ingreso: `https://sprl.sunarp.gob.pe/sprl/ingreso`
  - Pasarela de autorización (SSO): `https://im01-autorizacion-sprl-production.apps.paas.sunarp.gob.pe/v1/sunarp-services/im/autorizacion/login`
  - Módulo objetivo (Visualización de partidas): `https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral`
- **Mecanismo de acceso:** Autenticación mediante cuenta de usuario registrada (`SUNARP_USER` y `SUNARP_PASS`) protegida con desafío interactivo de **Cloudflare Turnstile**.
- **Resultado del módulo de autenticación:** Obtención y persistencia en disco (`sunarp_sprl_auth.json` + perfil nativo `./chrome_sprl_profile`) de las cookies de sesión (`cf_clearance`, cookies de aplicación) y del estado de `localStorage` (tokens JWT de los orígenes de SUNARP) para reutilizarlos en consultas posteriores sin repetir el login.

## Estrategia Técnica de Autenticación y Evasión Anti-Bot

- **Navegador con contexto persistente (`launch_persistent_context`):**
  1. Se levanta Google Chrome nativo (`channel="chrome"`) sobre un directorio de perfil dedicado (`./chrome_sprl_profile`), sin sobrescribir el `User-Agent` real del binario para mantener consistencia total con los *Client Hints* (`sec-ch-ua`) evaluados por Cloudflare.
  2. Se desactiva la bandera `--no-sandbox` por defecto de Playwright (`ignore_default_args=["--no-sandbox"]`) y se oculta `navigator.webdriver`.
- **Orquestación del flujo Single Sign-On (SSO):**
  1. Se navega inicialmente a la ruta privada objetivo (`/sprl/main/partidas-base-grafica-registral`). Si no existe sesión activa, el portal redirige a `/sprl/ingreso`.
  2. Se cierra automáticamente el modal publicitario emergente (*"Sunarp Simplifica"*) mediante evento `Escape`, clic en el botón de cierre o limpieza del contenedor overlay en el DOM, y se acciona el botón `INGRESAR` de la portada para generar el contexto de redirección SSO válido.
- **Resolución automatizada de Cloudflare Turnstile:**
  1. Se completan las credenciales con tipeo secuencial (`press_sequentially`) simulando cadencia humana.
  2. Se monitorea el `input[name="cf-turnstile-response"]` y, en caso de requerir interacción sobre la casilla *"Verifique que es un ser humano"*, se calcula dinámicamente la caja delimitadora (`bounding_box`) del widget para ejecutar un clic físico con trayectoria de ratón (`page.mouse.move` + `page.mouse.click`).
- **Persistencia de estado (`storage_state`):**
  - Una vez completada la redirección automática hacia `/sprl/main/partidas-base-grafica-registral`, se exportan las cookies y los orígenes (`localStorage`) a `sunarp_sprl_auth.json`.

In [7]:
import asyncio
import json
import os
from pathlib import Path
from playwright.async_api import async_playwright

URL_OBJETIVO_SPRL = "https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral"
URL_LOGIN_DIRECTO = "https://im01-autorizacion-sprl-production.apps.paas.sunarp.gob.pe/v1/sunarp-services/im/autorizacion/login"

SUNARP_USER = os.getenv("SUNARP_USER", "MYUSER123")
SUNARP_PASS = os.getenv("SUNARP_PASS", "BU&VQESCG9")

ARCHIVO_SESION = Path("sunarp_sprl_auth.json")

async def cerrar_modal_publicitario(page):
    """Cierra el modal emergente 'sunarp simplifica' en /sprl/ingreso si está presente."""
    await page.wait_for_timeout(1000)

    # 1. Intentar cerrar con tecla Escape (estándar en Angular Material / Bootstrap)
    await page.keyboard.press("Escape")
    await page.wait_for_timeout(400)

    # 2. Buscar directamente el botón 'X' de cierre del modal
    selectores_x = [
        "mat-dialog-container button:has-text('×')",
        "mat-dialog-container button:has-text('X')",
        ".modal-content button.close",
        "button[aria-label*='Close' i]",
        "button[aria-label*='Cerrar' i]",
        "span:has-text('×')",
        "div[role='dialog'] button"
    ]
    for sel in selectores_x:
        loc = page.locator(sel).first
        if await loc.count() > 0 and await loc.is_visible():
            try:
                await loc.click(timeout=1500)
                print("   -> Modal publicitario cerrado con botón 'X'.")
                await page.wait_for_timeout(500)
                return
            except Exception:
                pass

    # 3. Fallback: eliminar cualquier overlay de modal desde el DOM o clicar fuera (esquina superior izquierda)
    await page.mouse.click(15, 15)
    await page.evaluate("""
        () => {
            document.querySelectorAll('.cdk-overlay-container, .modal-backdrop, .modal, mat-dialog-container')
                .forEach(el => el.remove());
            document.body.style.overflow = 'auto';
        }
    """)
    await page.wait_for_timeout(300)

async def clicar_checkbox_turnstile(page) -> bool:
    selectores = [
        "iframe[src*='cloudflare']",
        "iframe[src*='turnstile']",
        "div.cf-turnstile",
        "#cf-turnstile"
    ]
    for sel in selectores:
        loc = page.locator(sel).first
        if await loc.count() > 0:
            box = await loc.bounding_box()
            if box:
                x = box["x"] + 32
                y = box["y"] + (box["height"] / 2)
                await page.mouse.move(x, y, steps=10)
                await page.wait_for_timeout(150)
                await page.mouse.click(x, y)
                return True

    btn = page.locator("button:has-text('INGRESAR')").first
    if await btn.count() > 0:
        box = await btn.bounding_box()
        if box:
            x = box["x"] + 35
            y = box["y"] - 55
            await page.mouse.move(x, y, steps=10)
            await page.wait_for_timeout(150)
            await page.mouse.click(x, y)
            return True
    return False

async def login_sunarp_sprl() -> dict:
    perfil_dir = Path("./chrome_sprl_profile")

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=str(perfil_dir),
            headless=False,
            channel="chrome",
            viewport={"width": 1366, "height": 768},
            locale="es-PE",
            ignore_default_args=["--no-sandbox"],
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-first-run",
                "--no-default-browser-check"
            ]
        )
        await context.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        page = context.pages[0] if context.pages else await context.new_page()

        try:
            print(f"[1/5] Navegando a la ruta objetivo: {URL_OBJETIVO_SPRL}...")
            await page.goto(URL_OBJETIVO_SPRL, wait_until="domcontentloaded", timeout=30000)
            await page.wait_for_timeout(2000)

            # Si redirigió a /sprl/ingreso (donde sale el modal 'sunarp simplifica'), cerramos el modal y damos clic en INGRESAR
            if "sprl/ingreso" in page.url:
                print("   -> Detectada portada '/sprl/ingreso'. Cerrando modal publicitario...")
                await cerrar_modal_publicitario(page)

                btn_portada = page.locator("button:has-text('INGRESAR'), a:has-text('INGRESAR')").first
                if await btn_portada.count() > 0:
                    print("   -> Clicando botón 'INGRESAR' de la portada para ir al login SSO...")
                    await btn_portada.click()
                    await page.wait_for_load_state("domcontentloaded")
                    await page.wait_for_timeout(1500)

            # Verificar si aún debemos llenar el formulario de credenciales
            texto_inicial = await page.inner_text("body")
            if "Hola," not in texto_inicial and "Visualización e impresión de partida" not in texto_inicial:
                input_user = page.locator("input[placeholder*='usuario' i], input[type='text']").first
                input_pass = page.locator("input[placeholder*='contraseña' i], input[type='password']").first

                await input_user.wait_for(state="visible", timeout=15000)
                print(f"[2/5] Ingresando credenciales de {SUNARP_USER}...")

                await input_user.click()
                await input_user.fill("")
                await input_user.press_sequentially(SUNARP_USER, delay=70)

                await input_pass.click()
                await input_pass.fill("")
                await input_pass.press_sequentially(SUNARP_PASS, delay=70)

                print("[3/5] Resolviendo verificación de Cloudflare Turnstile...")
                await page.wait_for_timeout(1500)

                for intento in range(20):
                    token_cf = await page.evaluate("""
                        () => {
                            const el = document.querySelector('input[name="cf-turnstile-response"], [name*="turnstile" i]');
                            return el && el.value ? el.value : "";
                        }
                    """)
                    if token_cf:
                        print(f"   -> ¡Operación exitosa! Token Turnstile capturado ({len(token_cf)} chars).")
                        break

                    if intento in (2, 8, 14):
                        print("   -> Haciendo clic automático en la casilla de verificación...")
                        await clicar_checkbox_turnstile(page)

                    await page.wait_for_timeout(1000)

                print("[4/5] Presionando botón 'INGRESAR'...")
                btn_ingresar = page.locator("button:has-text('INGRESAR'), input[value*='INGRESAR' i]").first
                await btn_ingresar.click()

                # Esperar a que redirija automáticamente de vuelta al portal SPRL autenticado
                await page.wait_for_selector(
                    "text=/Hola,|SALDO DISPONIBLE|BUSCAR SERVICIOS|Visualización e impresión de partida/i",
                    timeout=25000
                )

            # Si entró a /sprl/main general, navegamos a la vista específica de partidas-base-grafica-registral
            if "partidas-base-grafica-registral" not in page.url:
                print(f"   -> Entrando a la vista final: {URL_OBJETIVO_SPRL}...")
                await page.goto(URL_OBJETIVO_SPRL, wait_until="domcontentloaded", timeout=30000)
                await page.wait_for_selector("text=/Visualización e impresión de partida|Oficina Registral/i", timeout=20000)

            estado = await context.storage_state(path=str(ARCHIVO_SESION))
            print(f"[5/5] ¡Listo! Ubicado en '{page.url}' y sesión guardada en '{ARCHIVO_SESION}'.")

            await page.wait_for_timeout(3000)
            return {
                "authenticated": True,
                "user": SUNARP_USER,
                "current_url": page.url,
                "cookies_saved": len(estado.get("cookies", [])),
                "origins_saved": len(estado.get("origins", [])),
                "session_file": str(ARCHIVO_SESION.resolve())
            }

        except Exception as exc:
            return {
                "authenticated": False,
                "current_url": page.url,
                "error": str(exc)
            }
        finally:
            await context.close()

# Ejecutar en la celda
resultado_sprl = await login_sunarp_sprl()
print("\n================== RESULTADO LOGIN SUNARP SPRL ==================")
print(json.dumps(resultado_sprl, indent=2, ensure_ascii=False))

[1/5] Navegando a la ruta objetivo: https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral...
   -> Detectada portada '/sprl/ingreso'. Cerrando modal publicitario...
   -> Clicando botón 'INGRESAR' de la portada para ir al login SSO...
[2/5] Ingresando credenciales de MYUSER123...
[3/5] Resolviendo verificación de Cloudflare Turnstile...
   -> Haciendo clic automático en la casilla de verificación...
   -> ¡Operación exitosa! Token Turnstile capturado (773 chars).
[4/5] Presionando botón 'INGRESAR'...
[5/5] ¡Listo! Ubicado en 'https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral' y sesión guardada en 'sunarp_sprl_auth.json'.

================== RESULTADO LOGIN SUNARP SPRL ==================
{
  "authenticated": true,
  "user": "MYUSER123",
  "current_url": "https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral",
  "cookies_saved": 4,
  "origins_saved": 2,
  "session_file": "C:\\Users\\luisf\\Documents\\Work\\INDECOPI\\simec\\notebooks\\sunar